# NCBI gene symbol group id query

In [ ]:
import subprocess
import re
from tqdm import tqdm
import pandas as pd
import scanpy
import numpy as np

In [ ]:

df_rnaseq = pd.read_csv(r'G:\data\transcriptome\ABC\metadata\WMB-10X\20231215\gene.csv',index_col=0)
adata_h = scanpy.read_h5ad(r'G:\data\transcriptome\Human\a5463d8f-07df-4870-8cae-bc504de762c8.h5ad',backed='r')
df_merfish = pd.read_csv(r'G:\data\transcriptome\MERFISH\gene.csv',index_col=0)


In [ ]:
# NCBI datasets command-line tools: https://www.ncbi.nlm.nih.gov/datasets/docs/v2/command-line-tools/download-and-install/
path_ncbi_exe = r'E:\ChromeDownloads\datasets.exe'
gene_symbol_group_id_list_mouse = []
for i in tqdm(range(len(df_rnaseq))):
    gene_symbol = df_rnaseq['gene_symbol'].iloc[i]
    command = f'{path_ncbi_exe} summary gene symbol {gene_symbol} --taxon 10090'
    result = subprocess.run(command, capture_output=True, text=True)

    pattern = r'"gene_groups":\[\{"id":"(\d+)"'
    match = re.search(pattern, result.stdout)
    if match:
        gene_group_id = int(match.group(1))
    else:
        gene_group_id = None
    
    gene_symbol_group_id_list_mouse.append(gene_group_id)

100%|█████████████████████████████████████████████████████████████████████████| 32285/32285 [21:19:11<00:00,  2.38s/it]


In [ ]:
df_m_gsgi = pd.DataFrame()
df_m_gsgi.index = df_rnaseq['gene_symbol']
df_m_gsgi['gene_symbol_group_id'] = gene_symbol_group_id_list_mouse
df_m_gsgi.to_csv(r'G:\data\transcriptome\gsgi_mouse.csv')

In [ ]:
path_ncbi_exe = r'E:\ChromeDownloads\datasets.exe'
gene_symbol_group_id_list_human = []
for i in tqdm(range(len(adata_h.var['Gene']))):
    gene_symbol = adata_h.var['Gene'].iloc[i]
    command = f'{path_ncbi_exe} summary gene symbol {gene_symbol} --taxon human'
    result = subprocess.run(command, capture_output=True, text=True)

    pattern = r'"gene_groups":\[\{"id":"(\d+)"'
    match = re.search(pattern, result.stdout)
    if match:
        gene_group_id = int(match.group(1))
    else:
        gene_group_id = None
    
    gene_symbol_group_id_list_human.append(gene_group_id)
    

In [38]:
df_h_gsgi = pd.DataFrame()
df_h_gsgi.index = adata_h.var['Gene']
df_h_gsgi['gene_symbol_group_id'] = gene_symbol_group_id_list_human
df_h_gsgi.to_csv(r'G:\data\transcriptome\gsgi_human.csv')

# Orthology Determination
using NCBI command line tool and query gene symbol group id
https://doi.org/10.1093/nar/gkx1095

In [8]:
df_m_gsgi = pd.read_csv(r'G:\data\transcriptome\gsgi_mouse.csv',index_col=0)
df_h_gsgi = pd.read_csv(r'G:\data\transcriptome\gsgi_human.csv',index_col=0)


In [ ]:
m_h_gene_pair = []
for idx_m in range(df_m_gsgi.shape[0]):
    m_gene, m_gene_gid = df_m_gsgi.index[idx_m], df_m_gsgi.iloc[idx_m, 0]
    if np.isnan(m_gene_gid): 
        continue

    tmpdf = df_h_gsgi[df_h_gsgi['gene_symbol_group_id']==m_gene_gid]
    h_gene=None
    if len(tmpdf)==1:
        h_gene = tmpdf.index[0]
    elif len(tmpdf)>1:
        for i in range(len(tmpdf)):
            tmp_h_gene = tmpdf.index[i]
            if m_gene.upper()==tmp_h_gene.upper():
                h_gene == tmp_h_gene
    if h_gene is not None:
        m_h_gene_pair.append([m_gene,h_gene])
print(len(m_h_gene_pair))

15817


In [121]:
df_homo_gene = pd.DataFrame(m_h_gene_pair,columns=['mouse','human'])
df_homo_gene = df_homo_gene.drop_duplicates()
tmpdf = df_homo_gene[df_homo_gene['human'].isin(df_homo_gene['human'].value_counts()[df_homo_gene['human'].value_counts()>1].index)]
delete_idx_list = []
for i in range(len(tmpdf)):
    tmpm,tmph = tmpdf.iloc[i]
    if tmpm.upper()!=tmph.upper():
        delete_idx_list.append(tmpdf.index[i])
df_homo_gene = df_homo_gene.drop(index=delete_idx_list)
df_homo_gene = df_homo_gene.reset_index(drop=True,inplace=False)
df_homo_gene.to_csv(r'G:\data\transcriptome\gene_symbol_group_id_homologous_ncbi.csv')

# Select ROI regions (Cortex and Subregions) and save as new files

In [5]:
import pandas as pd
import scanpy as sc
import numpy as np
import anndata
import scipy.sparse as sp

In [ ]:
df_homo_gene = pd.read_csv(r'G:\data\transcriptome\gene_symbol_group_id_homologous.csv',index_col=0)

# A series of genes associated with mitochondria, heat-shock proteins, ribosomes, and dissociation
# Xue, R., Zhang, Q., Cao, Q., Kong, R., Xiang, X., Liu, H., ... & Zhang, N. (2022). Liver tumour immune microenvironment subtypes and neutrophil heterogeneity. Nature, 612(7938), 141-147.
df_excluded_gene = pd.read_csv(r'E:/ZhixiYun/Projects/fMOST_atlas/Data/gene/excluded_genes.txt', header=None)

In [16]:
# save excluded_gene "hgXXX" number
df_homo_gene[df_homo_gene.human.isin(df_excluded_gene.values.flatten())].to_csv(r'E:\ZhixiYun\Projects\fMOST_atlas\Data\gene\gene_symbol_group_id_homologous_excluded_genes.csv')


## Human Filtered Data

In [ ]:
# Talati A, Hirsch J (July 2005). "Functional specialization within the medial frontal gyrus for perceptual go/no-go decisions based on "what," "when," and "where" related information: an fMRI study". J Cogn Neurosci. 17 (7): 981–93. doi:10.1162/0898929054475226. PMID 16102231. S2CID 17680371.
# Data from https://doi.org/10.1126/science.add7046
# STable2 https://www.science.org/doi/abs/10.1126/science.add7046
br_label_dict = {
    'MFG':['Human A46'],
    'IFG':['Human A44-A45'],
    'SPL':['Human A5-A7'],
    'TP': ['Human A38'],
    'STG':['Human STG'],
    'MTG':['Human MTG']
}
used_cortical_regions = [f'Human {x}' for x in ['A1C','A23','A40','A44-A45','A46','A5-A7','A24','ITG','M1C','MTG','Pro','S1C','STG','TF','TH-TL','V1C','V2']]

In [ ]:
# Data Availability from https://doi.org/10.1126/science.add7046
adata_h = sc.read_h5ad(r'G:\data\transcriptome\Human\a5463d8f-07df-4870-8cae-bc504de762c8.h5ad', backed='r')

In [6]:
adata_h.obs[adata_h.obs.roi.isin(used_cortical_regions)].supercluster_term.value_counts()
cortical_neuron_clusters = [
    'Upper-layer intratelencephalic',
    'Deep-layer intratelencephalic',
    'MGE interneuron',
    'CGE interneuron',
    'Deep-layer corticothalamic and 6b',
    'LAMP5-LHX6 and Chandelier',
    'Deep-layer near-projecting',
    'Miscellaneous'
]

In [ ]:
df_filtered_genes = pd.read_csv(r'G:\data\transcriptome\Human\filtered_transcripts.txt',sep='\t',index_col=0)
# supplementary table 2 from https://doi.org/10.1126/science.add7046
df_filtered_10x_sample = pd.read_excel(r'G:\data\transcriptome\Human\science.add7046_table_s2.xlsx',index_col=0)

In [8]:
sample_10x_list = df_filtered_10x_sample.index[df_filtered_10x_sample['Sample retained in analysis']=='Y']
# roi_list = [y for x in br_label_dict.values() for y in x]
# row_mask = np.argwhere((adata_h.obs['roi'].isin(roi_list))&(adata_h.obs['sample_id'].isin(sample_10x_list))).flatten()
row_mask = np.argwhere((adata_h.obs['roi'].isin(used_cortical_regions))&(adata_h.obs['sample_id'].isin(sample_10x_list))&(adata_h.obs['supercluster_term'].isin(cortical_neuron_clusters))).flatten()
col_mask = np.argwhere(adata_h.var['Gene'].isin(df_homo_gene['human'])).flatten()
print(len(row_mask),len(col_mask))

654053 15506


In [9]:
adata_h_homo = adata_h[row_mask, col_mask]

In [10]:
adata_h_homo_csr_matrix = adata_h.X[row_mask]
adata_h_homo_csr_matrix = adata_h_homo_csr_matrix[:,col_mask]

In [11]:
adata_h_to_be_save = anndata.AnnData(adata_h_homo_csr_matrix)

In [12]:
adata_h_to_be_save.obs = adata_h_homo.obs
adata_h_to_be_save.var = adata_h_homo.var
adata_h_to_be_save.uns = adata_h_homo.uns

In [13]:
adata_h_to_be_save.write_h5ad(r'G:\data\transcriptome\Human\MH_homo_neurons.h5ad',compression='gzip')

## Mouse scRNA-seq  Filtered Data
https://www.nature.com/articles/s41586-023-06812-z

In [3]:
adata_m1 = sc.read_h5ad(r'G:\data\transcriptome\ABC\expression_matrices\WMB-10Xv3\20230630\WMB-10Xv3-Isocortex-1-raw.h5ad',)
adata_m2 = sc.read_h5ad(r'G:\data\transcriptome\ABC\expression_matrices\WMB-10Xv3\20230630\WMB-10Xv3-Isocortex-2-raw.h5ad',)

In [4]:
adata_m1.var['mt'] = adata_m1.var.gene_symbol.str.startswith("mt-")
adata_m2.var['mt'] = adata_m2.var.gene_symbol.str.startswith("mt-")
sc.pp.calculate_qc_metrics(adata_m1, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)
sc.pp.calculate_qc_metrics(adata_m2, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)

In [5]:
df_homo_gene = pd.read_csv(r'G:\data\transcriptome\gene_symbol_group_id_homologous.csv',index_col=0)

In [6]:
df_cell_metadata = pd.read_csv(r'G:\data\transcriptome\ABC\metadata\WMB-10X\20231215\cell_metadata.csv',index_col=0)
df_cell_metadata_isocortex = df_cell_metadata[df_cell_metadata['feature_matrix_label'].str.contains(r'WMB-10Xv3-Isocortex-\d')]
# df_cell_metadata_isocortex.to_csv(r'G:\data\transcriptome\ABC\cell_metadata_isocortex.csv')

In [7]:
meta_cell_list = np.array(df_cell_metadata_isocortex.index)
row_mask1 = np.argwhere((adata_m1.obs.index.isin(meta_cell_list))).flatten()
row_mask2 = np.argwhere((adata_m2.obs.index.isin(meta_cell_list))).flatten()
col_mask = np.argwhere(adata_m1.var['gene_symbol'].isin(df_homo_gene['mouse'])).flatten()
print(len(col_mask))

m1_obs = adata_m1.obs.iloc[row_mask1].copy()
m2_obs = adata_m2.obs.iloc[row_mask2].copy()
m_obs = pd.concat([m1_obs, m2_obs])
print(m_obs.shape)

adata_m1_homo = adata_m1[row_mask1,col_mask]

adata_m1_homo_csr_matrix = adata_m1.X[row_mask1]
adata_m1_homo_csr_matrix = adata_m1_homo_csr_matrix[:,col_mask]
adata_m2_homo_csr_matrix = adata_m2.X[row_mask2]
adata_m2_homo_csr_matrix = adata_m2_homo_csr_matrix[:,col_mask]

adata_m_to_be_save = anndata.AnnData(sp.vstack((adata_m1_homo_csr_matrix,adata_m2_homo_csr_matrix), format='csr'))
adata_m_to_be_save.obs = m_obs
adata_m_to_be_save.var = adata_m1_homo.var

15506
(455207, 7)


In [8]:
import gc
del adata_m1_homo_csr_matrix
del adata_m2_homo_csr_matrix
gc.collect()

0

In [9]:
adata_m_to_be_save.write_h5ad(r'G:\data\transcriptome\ABC\MH_m_isocortex_neurons.h5ad', compression='gzip')

## MOUSE MERFISH Filtered Data
https://www.nature.com/articles/s41586-023-06808-9

In [7]:
import sys
sys.path.append(r'E:\ZhixiYun\Projects\Neuron_Morphology_Table\Scripts')
from basicfunc import MouseAnatomyTree
import gc

In [16]:
adatas_m_merfish = [sc.read_h5ad(r'G:\data\transcriptome\MERFISH\WB_MERFISH_animal1_coronal.h5ad').raw.to_adata(),
                    sc.read_h5ad(r'G:\data\transcriptome\MERFISH\WB_MERFISH_animal2_coronal.h5ad').raw.to_adata(),
                    sc.read_h5ad(r'G:\data\transcriptome\MERFISH\WB_MERFISH_animal3_sagittal.h5ad').raw.to_adata(),
                    sc.read_h5ad(r'G:\data\transcriptome\MERFISH\WB_MERFISH_animal4_sagittal.h5ad').raw.to_adata(),
                   ] 

In [17]:
col_mask = adatas_m_merfish[0].var['gene_name'].isin(df_homo_gene['mouse'])
print('#. keep genes: ',np.sum(col_mask))

neuron_list = ['GABAergic neuron', 'cholinergic neuron', 'dopaminergic neuron',
               'glutamatergic neuron', 'glycinergic neuron', 'histaminergic neuron']

# add brain region
mouse_ct_lut_tmp = {
    # 'OL':['VISp','VISam','VISpm','VISl','VISli','VISpl'],
   'FL':['MOp','MOs','ORB','FRP'],
   'PL':['SSp','SSs','PTLp','VISC'],
   'TL':['AUDp','AUDd','AUDv','AUDpo','PERI','TEa','ECT','VISpor']
}
roi_ct_list = [v for k in mouse_ct_lut_tmp.values() for v in k]
given_ct_list = ['MOp','MOs','ORB','FRP', 'SSp','SSs','PTLp','VISC', 'AUD','PERI','TEa','ECT', 'RSP', 'AI', 'ILA', 'PL','ACA','VIS','GU']
MAT = MouseAnatomyTree()
lutfull2name = dict(zip([x['name'] for x in MAT.tree], [x['acronym'] for x in MAT.tree]))

for i in range(4):
    adata = adatas_m_merfish[i]
    rgn_list = adata.obs['ccf_region_name'].apply(lutfull2name.get)
    rgn_nolayer_list = MAT.ctlist_to_given_ctlist(rgn_list, given_ct_list, not_in_set_None=True)
    rgn_roi_list = MAT.ctlist_to_given_ctlist(rgn_list, roi_ct_list, not_in_set_None=True)
    rough_rgn_list = MAT.ctlist_to_given_ctlist(rgn_list, ['Isocortex'], not_in_set_None=True)
    
    adata.obs['region'] = rgn_list
    adata.obs['region_nolayer'] = rgn_nolayer_list
    adata.obs['region_roi'] = rgn_roi_list
    adata.obs['rough_region'] = rough_rgn_list

    adatas_m_merfish[i] = adata[(adata.obs['rough_region']=='Isocortex')&(adata.obs['cell_type'].isin(neuron_list)), col_mask].copy()
    
    print(adatas_m_merfish[i].shape)
    

#. keep genes:  1013
(575494, 1013)
(267919, 1013)
(279594, 1013)
(43572, 1013)


In [19]:
adata_m_merfish = adatas_m_merfish[0]
for i in range(1, 4):
    adata_m_merfish = anndata.concat([adata_m_merfish, adatas_m_merfish[i]],merge='same')
print(adata_m_merfish)

del adatas_m_merfish[1:]
gc.collect()

adata_m_merfish.var['ori_index'] = adata_m_merfish.var.index
gene_name_2_hg = dict(zip(df_homo_gene.mouse, df_homo_gene.acronym))
adata_m_merfish.var.index = [gene_name_2_hg[x] for x in adata_m_merfish.var.gene_name]


AnnData object with n_obs × n_vars = 1166579 × 1013
    obs: 'organism_ontology_term_id', 'donor_id', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'disease_ontology_term_id', 'tissue_ontology_term_id', 'cell_type_ontology_term_id', 'assay_ontology_term_id', 'suspension_type', 'cluster_id_transfer', 'subclass_transfer', 'cluster_confidence_score', 'subclass_confidence_score', 'high_quality_transfer', 'major_brain_region', 'ccf_region_name', 'brain_section_label', 'tissue_type', 'is_primary_data', 'cell_type', 'assay', 'disease', 'organism', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'region', 'region_nolayer', 'region_roi', 'rough_region'
    var: 'gene_name', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
    obsm: 'X_CCF', 'X_spatial_coords', 'X_umap'


In [20]:
adata_m_merfish.write_h5ad(r'G:\data\transcriptome\MERFISH\MH_m_merfish_isocortex_neurons.h5ad', compression='gzip')

In [21]:
adata_m_merfish.obs.to_csv(r'G:\data\transcriptome\MERFISH\MH_m_merfish_isocortex_neurons_obs.csv')
adata_m_merfish.var.to_csv(r'G:\data\transcriptome\MERFISH\MH_m_merfish_isocortex_neurons_var.csv')